# 1.1. Khám phá danh mục bài báo và lịch sử phiên bản

## 1. Giao thức truy vấn Atom XML và OAI-PMH

1. **Truy vấn theo lô qua Atom XML (`export.arxiv.org/api/query`)**: Module `discovery.py` sử dụng `requests.Session` để truy vấn danh sách `id_list` từ 50 đến 100 mã mỗi lượt với thời gian nghỉ tối thiểu 3,0 giây giữa các yêu cầu, đồng thời kiểm tra các bản ghi lỗi trong luồng XML và đọc hậu tố phiên bản mới nhất (`vN`) để sinh đầy đủ danh sách các phiên bản từ v1 đến vN cho từng bài báo.
2. **Tìm kiếm nhị phân biên tháng và đối soát qua `oaipmh.arxiv.org`**: Hai phân vùng thành viên nằm vắt qua ranh giới tháng (từ tháng 12/2024 sang 01/2025 của DAT và từ tháng 06/2024 sang 07/2024 của TRI). Hàm `id_range.py:find_last_id_of_month` dùng tìm kiếm nhị phân xác định mã số cuối cùng của tháng và gọi `discovery.py:confirm_missing_id` tới địa chỉ OAI-PMH mới `https://oaipmh.arxiv.org/oai` (`metadataPrefix=arXivRaw`, áp dụng từ đợt nâng cấp hạ tầng tháng 03/2025 của arXiv) nhằm phân biệt rõ mã `idDoesNotExist` với lỗi mạng tạm thời, đồng thời đối chiếu ngày nộp từng phiên bản trong `version_history.py`.

## 2. Phân bố số phiên bản và tỷ lệ khuyết trường thông tin

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="dataframe ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
hist_s = pd.Series({int(k): int(v) for k, v in m1["version_histogram"].items()}).sort_index()
df_hist = pd.DataFrame({
    "Số phiên bản / bài": hist_s.index,
    "Số bài báo": hist_s.values,
})
df_hist["Tỷ lệ (%)"] = 100.0 * df_hist["Số bài báo"] / scope["papers"]
df_hist["Tích lũy (%)"] = df_hist["Tỷ lệ (%)"].cumsum()
display(style_table(
    df_hist,
    caption="Bảng 1.1. Phân bố số phiên bản trên 25.000 bài báo.",
    int_cols=["Số phiên bản / bài", "Số bài báo"],
    pct_cols=["Tỷ lệ (%)", "Tích lũy (%)"],
))

fig, ax1 = plt.subplots(figsize=(9.8, 4.0))
x = np.arange(len(df_hist))
bars = ax1.bar(x, df_hist["Số bài báo"].to_numpy(dtype=float), color=BOOK_PALETTE["navy"], width=0.64)
ax1.set_xticks(x)
ax1.set_xticklabels([str(int(v)) for v in df_hist["Số phiên bản / bài"]], rotation=0)
ax1.set_xlabel("Số phiên bản của một bài báo")
ax1.set_ylabel("Số bài báo")
ax1.set_ylim(0, float(df_hist["Số bài báo"].max()) * 1.18)
ax1.grid(axis="y")
for idx_b, (b, cnt, pct) in enumerate(zip(bars, df_hist["Số bài báo"], df_hist["Tỷ lệ (%)"])):
    if idx_b < 5:
        ax1.text(b.get_x() + b.get_width() / 2, float(cnt) + 220, f"{fmt_int(cnt)}\n({fmt_pct(pct, 1)})",
                 ha="center", va="bottom", fontsize=7.7)

ax2 = ax1.twinx()
ax2.spines["top"].set_visible(False)
ax2.plot(x, df_hist["Tích lũy (%)"].to_numpy(dtype=float), color=BOOK_PALETTE["steel"], marker="o", lw=1.8, markersize=4.2)
ax2.set_ylabel("Tỷ lệ tích lũy (%)", color=BOOK_PALETTE["steel"])
ax2.set_ylim(50, 103)

ins = ax1.inset_axes([0.46, 0.36, 0.46, 0.46])
tail = df_hist[df_hist["Số phiên bản / bài"] >= 5]
tx = np.arange(len(tail))
ins.bar(tx, tail["Số bài báo"].to_numpy(dtype=float), color=BOOK_PALETTE["slate"], width=0.62)
ins.set_xticks(tx)
ins.set_xticklabels([str(int(v)) for v in tail["Số phiên bản / bài"]])
ins.set_yscale("log")
ins.set_title("Nhóm từ 5 đến 17 phiên bản (thang log)", fontsize=8.2)
ins.tick_params(labelsize=7.5)
for i_t, v_t in enumerate(tail["Số bài báo"]):
    ins.text(i_t, float(v_t) * 1.18, str(int(v_t)), ha="center", fontsize=7.1)

ax1.set_title("Hình 1.1. Phân bố số phiên bản trên mỗi bài báo và đường tích lũy (trung bình 1,66 phiên bản/bài)")
plt.tight_layout()
plt.show()

In [ ]:
miss_m1 = pd.Series(m1["metadata_missing_fraction"])
df_meta = pd.DataFrame({
    "Trường metadata": [str(k) for k in miss_m1.index],
    "Tỷ lệ có giá trị (%)": (1.0 - miss_m1.values) * 100.0,
    "Tỷ lệ khuyết (%)": miss_m1.values * 100.0,
})
display(style_table(
    df_meta,
    caption="Bảng 1.2. Tỷ lệ đầy đủ của các trường metadata cấp bài báo trên 25.000 bài.",
    pct_cols=["Tỷ lệ có giá trị (%)", "Tỷ lệ khuyết (%)"],
))

fig, ax = plt.subplots(figsize=(9.2, 2.8))
y = np.arange(len(df_meta))
v_ok = df_meta["Tỷ lệ có giá trị (%)"].to_numpy(dtype=float)
v_na = df_meta["Tỷ lệ khuyết (%)"].to_numpy(dtype=float)
ax.barh(y, v_ok, color=BOOK_PALETTE["navy"], height=0.50, label="Có giá trị (%)")
ax.barh(y, v_na, left=v_ok, color="#E2E8F0", edgecolor="#94A3B8", height=0.50, label="Khuyết trên Atom feed (%)")
ax.set_yticks(y)
ax.set_yticklabels([str(v) for v in df_meta["Trường metadata"]])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Tỷ lệ (%)")
ax.set_title("Hình 1.2. Độ đầy đủ các trường thông tin cấp bài báo (giai đoạn 06/2024 - 02/2025)")
ax.legend(loc="lower right", frameon=True, fontsize=8.2)
for idx_m, row_m in enumerate(df_meta.to_dict("records")):
    full_v = row_m["Tỷ lệ có giá trị (%)"]
    if full_v > 50:
        ax.text(full_v / 2, idx_m, fmt_pct(full_v), ha="center", va="center", color="white", fontweight="bold", fontsize=8.2)
    else:
        ax.text(full_v + 2.0, idx_m, f"Có giá trị {fmt_pct(full_v)} | Khuyết {fmt_pct(row_m['Tỷ lệ khuyết (%)'])}",
                va="center", color="#1E293B", fontsize=8.1)
plt.tight_layout()
plt.show()

```{note}
**Giải thích số lượng 5.565 mốc ngày phiên bản trung gian mang giá trị rỗng (`null`):**
Trên 25.000 bài báo có 14.067 bài chỉ gồm 1 phiên bản (v1 nhận mốc `atom:published`) và 10.933 bài có từ 2 phiên bản trở lên (v1 nhận `published`, vN nhận `updated`). Tổng số mốc thời gian mà Atom XML cung cấp trực tiếp là $14.067 \times 1 + 10.933 \times 2 = 35.933$ phiên bản. Phần chênh lệch $41.498 - 35.933 = 5.565$ khớp chính xác với các phiên bản trung gian $v_2 \dots v_{N-1}$ của 3.667 bài báo có từ 3 phiên bản trở lên (hệ thống giữ nguyên `null`, không nội suy ngày giả). Tương tự, trường `doi` khuyết 77,70% và `venue` khuyết 82,54% do phần lớn bản thảo mới nộp giai đoạn 2024-2025 chưa qua xuất bản tạp chí chính thức.
```

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 1 - Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/id_range.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/id_range.py).
- Mã nguồn triển khai: [`src/crawler/discovery.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/discovery.py).
- Mã nguồn triển khai: [`src/crawler/version_history.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/version_history.py).
- Mã nguồn triển khai: [`src/crawler/request_policy.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/request_policy.py).
- Mã nguồn triển khai: [`src/crawler/rate_limiter.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/rate_limiter.py).